# Using NPF through Python with enoslib

First let's go through the traditional enoslib setup. In this case instead of reserving cluster node, we'll use the localhost Host. 
At the end of this notebook you'll find an example using Grid5K.

In [ ]:
!pip install enoslib

import enoslib as en
import logging

# Display some general information about the library
en.check()

# Enable rich logging
_ = en.init_logging()

roles={"client":en.Host("localhost", user="tbarbette")}


#en.set_config(ansible_stdout="regular")
en.run_command("date", roles=roles["client"])

## Launch NPF

Simply run an NPF script using the snippet below. This will run the "math.npf" test which computes the log and power of the value N with N from 1 to 50 with a given POWER parameter. Look at math.npf for detail.

In [ ]:
from npf import enoslib as npf


results,_ = npf.run("math.npf",
        series=["local+POWER=1:Linear","local+POWER=2:Quadratic"], #List of repositories, local means no repository (locally installed software). POWER=2 allows to overwrite variable for a given serie.
        roles=roles)

display(results)

## Get a panda dataset 

In [ ]:
from npf.output.transform.pandas import to_pandas


df = to_pandas(results)

display(df)

# Grid5K example

In [ ]:
import enoslib as en
from npf import enoslib as npf
from npf.output.transform.pandas import to_pandas

from importlib import reload
reload(npf)

conf = (
    en.G5kConf.from_settings(job_name="npf-test", walltime="0:30:00")
    .add_machine(roles=["client"], cluster="spirou", nodes=1) #gros is a cluster on nancy
)

print("Reserving resources...")
# This will validate the configuration, but not reserve resources yet
provider = en.G5k(conf)

# Get actual resources
roles, networks = provider.init()

print("Launching NPF")
results, _ = npf.run("math.npf",
        series=["local+POWER=1:Linear","local+POWER=2:Quadratic"], #List of repositories, local means no repository (locally installed software). POWER=2 allows to overwrite variable for a given serie.
        roles=roles)
df = to_pandas(results)

display(df)